In [ ]:
"""
Transform a raw ClinicalTrials.gov API study dict into normalized records
ready for BigQuery insertion.

Returns a dict with keys mapping to target tables:
  - 'trial':         the main row for clinical_trials_gov_stg
  - 'locations':     rows for trial_locations
  - 'mesh_terms':    rows for trial_mesh_terms
  - 'arm_groups':    rows for trial_arm_groups
  - 'interventions': rows for trial_interventions
  - 'outcomes':      rows for trial_outcomes
  - 'officials':     rows for trial_officials

Returns None only when nct_id is missing.
"""

import logging
import re
from datetime import date, datetime
from typing import Optional
from typing import Any, Optional

from google.auth import default

logger = logging.getLogger(__name__)


# ─── Helpers ─────────────────────────────────────────────────────────────────


def _get(d: dict, *keys, default=None):
    """Safely traverse nested dicts; returns `default` on any missing key."""
    current: Any = d
    for k in keys:
        if not isinstance(current, dict):
            return default
        current = current.get(k)
        if current is None:
            return default
    return current if current is not None else default


def _parse_date(date_struct: Optional[dict]) -> Optional[date]:
    """
    Parse a CT.gov dateStruct into a Python date.

    Handles full dates (YYYY-MM-DD), year-month (YYYY-MM), and year-only (YYYY).
    Partial dates default to the 1st of the month / January 1st.
    Returns None on missing or unparseable input.
    """
    if not date_struct:
        return None
    raw = date_struct.get("date") if isinstance(date_struct, dict) else date_struct
    if not raw:
        return None
    for fmt in ("%Y-%m-%d", "%Y-%m", "%Y"):
        try:
            return datetime.strptime(raw, fmt).date()
        except ValueError:
            continue
    logger.warning("Unparseable date: %r", raw)
    return None


_AGE_RE = re.compile(r"^\s*(\d+(?:\.\d+)?)\s+(\w+)", re.IGNORECASE)
_AGE_UNIT_TO_YEARS = {
    "year": 1.0, "years": 1.0,
    "month": 1 / 12, "months": 1 / 12,
    "week": 1 / 52.143, "weeks": 1 / 52.143,
    "day": 1 / 365.25, "days": 1 / 365.25,
    "hour": 1 / (365.25 * 24), "hours": 1 / (365.25 * 24),
    "minute": 1 / (365.25 * 24 * 60), "minutes": 1 / (365.25 * 24 * 60),
}


def _parse_age_to_years(age_str: Optional[str]) -> Optional[float]:
    """Convert strings like '30 Years', '6 Months', '2 Weeks' to years (float)."""
    if not age_str or not isinstance(age_str, str):
        return None
    m = _AGE_RE.match(age_str.strip())
    if not m:
        return None
    value, unit = m.groups()
    multiplier = _AGE_UNIT_TO_YEARS.get(unit.lower())
    if multiplier is None:
        logger.debug("Unknown age unit: %r", unit)
        return None
    return float(value) * multiplier


# ─── Child-table extractors ──────────────────────────────────────────────────

def _extract_locations(protocol: dict, nct_id: str, last_update: Optional[date]) -> list[dict]:
    """One row per study site. Powers geo / matching queries."""
    locations = _get(protocol, "contactsLocationsModule", "locations", default=[]) or []
    out = []
    print(locations)
    for loc in locations:
        geo = loc.get("geoPoint") or {}
        out.append({
            "nct_id": nct_id,
            "facility": loc.get("facility"),
            "city": loc.get("city"),
            "state": loc.get("state"),
            "country": loc.get("country"),
            "zip": loc.get("zip"),
            "status": loc.get("status"),  # per-site recruiting status
            "latitude": geo.get("lat"),
            "longitude": geo.get("lon"),
            "last_update_post_date": last_update,  # for partitioning and incremental updates
            "contacts": [
                {
                    "name": c.get("name"),
                    "role": c.get("role"),
                    "phone": c.get("phone"),
                    "email": c.get("email"),
                }
                for c in (loc.get("contacts") or [])
            ],
        })
    return out


def _extract_mesh_terms(derived: dict, nct_id: str, last_update: Optional[date]) -> list[dict]:
    """One row per MeSH term, flagged with browse_type and ancestor status."""
    out = []
    for browse_module_key in ("conditionBrowseModule", "interventionBrowseModule"):
        module = derived.get(browse_module_key) or {}
        browse_type = "condition" if browse_module_key == "conditionBrowseModule" else "intervention"
        for term in (module.get("meshes") or []):
            out.append({
                "nct_id": nct_id,
                "mesh_id": term.get("id"),
                "mesh_term": term.get("term"),
                "browse_type": browse_type,
                "is_ancestor": False,
                "last_update_post_date": last_update,
            })
        for term in (module.get("ancestors") or []):
            out.append({
                "nct_id": nct_id,
                "mesh_id": term.get("id"),
                "mesh_term": term.get("term"),
                "browse_type": browse_type,
                "is_ancestor": True,
                "last_update_post_date": last_update,
            })
    return out


def _extract_arm_groups(protocol: dict, nct_id: str, last_update: Optional[date]) -> list[dict]:
    arms = _get(protocol, "armsInterventionsModule", "armGroups", default=[]) or []
    return [
        {
            "nct_id": nct_id,
            "label": a.get("label"),
            "type": a.get("type"),
            "description": a.get("description"),
            "intervention_names": a.get("interventionNames") or [],
            "last_update_post_date": last_update,
        }
        for a in arms
    ]


def _extract_interventions(protocol: dict, nct_id: str, last_update: Optional[date]) -> list[dict]:
    interventions = _get(protocol, "armsInterventionsModule", "interventions", default=[]) or []
    return [
        {
            "nct_id": nct_id,
            "type": iv.get("type"),
            "name": iv.get("name"),
            "description": iv.get("description"),
            "other_names": iv.get("otherNames") or [],
            "arm_group_labels": iv.get("armGroupLabels") or [],
            "last_update_post_date": last_update,
        }
        for iv in interventions
    ]


def _extract_outcomes(protocol: dict, nct_id: str, last_update: Optional[date]) -> list[dict]:
    """Unified outcomes table: primary / secondary / other in one place."""
    out = []
    outcomes_mod = protocol.get("outcomesModule") or {}
    for kind, key in (("primary", "primaryOutcomes"),
                      ("secondary", "secondaryOutcomes"),
                      ("other", "otherOutcomes")):
        for o in (outcomes_mod.get(key) or []):
            out.append({
                "nct_id": nct_id,
                "outcome_type": kind,
                "measure": o.get("measure"),
                "description": o.get("description"),
                "time_frame": o.get("timeFrame"),
                "last_update_post_date": last_update,
            })
    return out


def _extract_officials(protocol: dict, nct_id: str, last_update: Optional[date]) -> list[dict]:
    """All PIs / study chairs / directors, not just the responsible party."""
    officials = _get(protocol, "contactsLocationsModule", "overallOfficials", default=[]) or []
    return [
        {
            "nct_id": nct_id,
            "name": o.get("name"),
            "affiliation": o.get("affiliation"),
            "role": o.get("role"),
            "last_update_post_date": last_update,
        }
        for o in officials
    ]


def _extract_collaborators(protocol: dict) -> list[dict]:
    """Keep the class (NIH / INDUSTRY / OTHER), not just the name."""
    collaborators = _get(protocol, "sponsorCollaboratorsModule", "collaborators", default=[]) or []
    print(collaborators)
    return [
        {"name": c.get("name"), "class": c.get("class")}
        for c in collaborators
        if c.get("name")
    ]


def _extract_references(protocol: dict) -> list[dict]:
    refs = _get(protocol, "referencesModule", "references", default=[]) or []
    out = []
    for r in refs:
        pmid = r.get("pmid")
        try:
            pmid_int = int(pmid) if pmid is not None else None
        except (TypeError, ValueError):
            pmid_int = None
        out.append({
            "citation": r.get("citation"),
            "type": r.get("type"),
            "pmid": pmid_int,
        })
    return out


def _extract_secondary_ids(protocol: dict) -> list[dict]:
    ids = _get(protocol, "identificationModule", "secondaryIdInfos", default=[]) or []
    return [
        {"id": i.get("id"), "type": i.get("type"), "link": i.get("link")}
        for i in ids
    ]


def _extract_central_contacts(protocol: dict) -> list[dict]:
    contacts = _get(protocol, "contactsLocationsModule", "centralContacts", default=[]) or []
    print(contacts)
    return [
        {
            "name": c.get("name"),
            "role": c.get("role"),
            "phone": c.get("phone"),
            "email": c.get("email"),
        }
        for c in contacts
    ]


# ─── Main transform ──────────────────────────────────────────────────────────

def transform_study(raw_study: dict) -> Optional[dict]:
    """
    Map one raw CT.gov study dict to a structured record set across multiple
    target tables.

    Returns None only when nct_id is missing (the one hard required field).
    Missing dates, sponsor info, etc. are allowed through as NULL.
    """
    protocol = raw_study.get("protocolSection") or {}
    derived = raw_study.get("derivedSection") or {}

    # ── Identification ─────────────────────────────────────────────────────
    id_mod = protocol.get("identificationModule") or {}
    nct_id: Optional[str] = id_mod.get("nctId")
    if not nct_id:
        logger.warning("Skipping study with no nctId")
        return None

    # ── Status / Dates ─────────────────────────────────────────────────────
    status_mod = protocol.get("statusModule") or {}
    # Note: no longer skip on missing startDate; allow NULL downstream.
    start_date = _parse_date(status_mod.get("startDateStruct"))
    last_update = _parse_date(status_mod.get("lastUpdatePostDateStruct"))
    if last_update is None:
        logger.debug("%s: missing lastUpdatePostDate", nct_id)

    # ── Sponsor / Collaborators ────────────────────────────────────────────
    sponsor_mod = protocol.get("sponsorCollaboratorsModule") or {}
    lead_sponsor = sponsor_mod.get("leadSponsor") or {}
    responsible_party = sponsor_mod.get("responsibleParty") or {}

    # ── Design ─────────────────────────────────────────────────────────────
    design_mod = protocol.get("designModule") or {}
    design_info = design_mod.get("designInfo") or {}
    masking_info = design_info.get("maskingInfo") or {}
    enrollment_info = design_mod.get("enrollmentInfo") or {}

    # ── Eligibility ────────────────────────────────────────────────────────
    elig_mod = protocol.get("eligibilityModule") or {}
    min_age_raw = elig_mod.get("minimumAge")
    max_age_raw = elig_mod.get("maximumAge")

    # ── Description / Conditions / Oversight ──────────────────────────────
    desc_mod = protocol.get("descriptionModule") or {}
    cond_mod = protocol.get("conditionsModule") or {}
    oversight_mod = protocol.get("oversightModule") or {}

    # ── Main trial row ─────────────────────────────────────────────────────
    trial = {
        # Identification
        "nct_id": nct_id,
        "brief_title": id_mod.get("briefTitle"),
        "official_title": id_mod.get("officialTitle"),
        "acronym": id_mod.get("acronym"),
        "organization": _get(id_mod, "organization", "fullName"),
        "organization_class": _get(id_mod, "organization", "class"),
        "secondary_ids": _extract_secondary_ids(protocol),

        # Status / Dates
        "overall_status": status_mod.get("overallStatus"),
        "has_results": raw_study.get("hasResults", False),
        "start_date": start_date,
        "start_date_type": _get(status_mod, "startDateStruct", "type"),
        "completion_date": _parse_date(status_mod.get("completionDateStruct")),
        "primary_completion_date": _parse_date(status_mod.get("primaryCompletionDateStruct")),
        "study_first_post_date": _parse_date(status_mod.get("studyFirstPostDateStruct")),
        "last_update_post_date": last_update,

        # Sponsor
        "lead_sponsor_name": lead_sponsor.get("name"),
        "lead_sponsor_class": lead_sponsor.get("class"),
        "collaborators": _extract_collaborators(protocol),
        "responsible_party_investigator": responsible_party.get("investigatorFullName"),
        "responsible_party_affiliation": responsible_party.get("investigatorAffiliation"),
        "responsible_party_type": responsible_party.get("type"),

        # Design
        "study_type": design_mod.get("studyType"),
        "phases": design_mod.get("phases") or [],
        "allocation": design_info.get("allocation"),
        "intervention_model": design_info.get("interventionModel"),
        "primary_purpose": design_info.get("primaryPurpose"),
        "masking": masking_info.get("masking"),
        "enrollment_count": enrollment_info.get("count"),
        "enrollment_type": enrollment_info.get("type"),  # ESTIMATED vs ACTUAL

        # Conditions / Keywords
        "conditions": cond_mod.get("conditions") or [],
        "keywords": cond_mod.get("keywords") or [],

        # Eligibility (raw text + structured for filters)
        "eligibility_criteria": elig_mod.get("eligibilityCriteria"),
        "min_age_raw": min_age_raw,
        "max_age_raw": max_age_raw,
        "min_age_years": _parse_age_to_years(min_age_raw),
        "max_age_years": _parse_age_to_years(max_age_raw),
        "sex": elig_mod.get("sex"),
        "std_ages": elig_mod.get("stdAges") or [],
        "healthy_volunteers": elig_mod.get("healthyVolunteers"),

        # Description
        "brief_summary": desc_mod.get("briefSummary"),
        "detailed_description": desc_mod.get("detailedDescription"),

        # Oversight / Regulatory
        "is_fda_regulated_drug": oversight_mod.get("isFdaRegulatedDrug"),
        "is_fda_regulated_device": oversight_mod.get("isFdaRegulatedDevice"),
        "has_dmc": oversight_mod.get("oversightHasDmc"),

        # Contacts (central, not site-specific — site contacts live in locations)
        "central_contacts": _extract_central_contacts(protocol),

        # Publications / references
        "references": _extract_references(protocol),
    }

    return {
        "trial": trial,
        "locations": _extract_locations(protocol, nct_id, last_update),
        "mesh_terms": _extract_mesh_terms(derived, nct_id, last_update),
        "arm_groups": _extract_arm_groups(protocol, nct_id, last_update),
        "interventions": _extract_interventions(protocol, nct_id, last_update),
        "outcomes": _extract_outcomes(protocol, nct_id, last_update),
        "officials": _extract_officials(protocol, nct_id, last_update),
    }

In [2]:
import json
from typing import Any

import requests

# Define the API endpoint and parameters
# This example searches for studies related to 'asthma'
url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "pageSize": 1,
    "filter.ids": "NCT00276835"  # Number of results per page
}

response = requests.get(url, params=params)
response.raise_for_status()  # Check for HTTP errors
print(json.dumps(response.json()))  # Print the raw JSON response for debugging
data = response.json()# Print the retrieved data
page_studies = data.get("studies", [])

{"studies": [{"protocolSection": {"identificationModule": {"nctId": "NCT00276835", "orgStudyIdInfo": {"id": "NU 04V1"}, "secondaryIdInfos": [{"id": "NU-04V1"}, {"id": "CHIR-NU-04V1"}, {"id": "NU-IRB-0310-083"}], "organization": {"fullName": "Northwestern University", "class": "OTHER"}, "briefTitle": "Genistein and Interleukin-2 in Treating Patients With Metastatic Melanoma or Kidney Cancer", "officialTitle": "A Pilot Study of the Effect of Genistein in Combination With High-Dose Interleukin-2 on Cell Expansion and Gene Expression in Patients With Metastatic Melanoma or Renal Cell Carcinoma"}, "statusModule": {"statusVerifiedDate": "2015-04", "overallStatus": "COMPLETED", "expandedAccessInfo": {"hasExpandedAccess": false}, "startDateStruct": {"date": "2005-11"}, "primaryCompletionDateStruct": {"date": "2007-07", "type": "ACTUAL"}, "completionDateStruct": {"date": "2014-01", "type": "ACTUAL"}, "studyFirstSubmitDate": "2006-01-12", "studyFirstSubmitQcDate": "2006-01-12", "studyFirstPostDa

In [40]:
transformed_study = transform_study(page_studies[0])

[{'name': 'National Cancer Institute (NCI)', 'class': 'NIH'}]
[]
[{'facility': 'Robert H. Lurie Comprehensive Cancer Center at Northwestern University', 'city': 'Chicago', 'state': 'Illinois', 'zip': '60611-3013', 'country': 'United States', 'geoPoint': {'lat': 41.85003, 'lon': -87.65005}}]


In [41]:
"""
Convert transformed clinical trials records into Parquet files and upload to
GCS in a Hive-partitioned layout that BigQuery can load directly.
 
Pipeline:
  transform_study()         (per study)
       │
       ▼
  list[dict]                (batched in memory)
       │
       ▼
  studies_to_arrow_tables() (one pa.Table per target)
       │
       ▼
  write_tables_to_gcs()     (Parquet upload, partitioned by ingestion_date)
 
GCS layout:
  gs://{bucket}/{prefix}/{table}/ingestion_date=YYYY-MM-DD/batch_{id}.parquet
 
  Example:
    gs://my-data-lake/clinical_trials/staging/trial/
        ingestion_date=2026-05-22/batch_a1b2c3d4e5f6.parquet
 
Each table directory follows Hive partitioning so BigQuery external tables
or LOAD jobs with `hive_partition_uri_prefix` pick up `ingestion_date`
automatically.
 
Required packages:
    pip install pyarrow google-cloud-storage
"""
from __future__ import annotations
 
import io
import logging
import uuid
from datetime import date, datetime, timezone
from typing import Iterable, Optional
 
import pyarrow as pa
import pyarrow.parquet as pq
from google.cloud import storage
 
logger = logging.getLogger(__name__)
 
 
# ─── Reusable nested struct types ────────────────────────────────────────────
 
_CONTACT_STRUCT = pa.struct([
    pa.field("name", pa.string()),
    pa.field("role", pa.string()),
    pa.field("phone", pa.string()),
    pa.field("email", pa.string()),
])
 
_SECONDARY_ID_STRUCT = pa.struct([
    pa.field("id", pa.string()),
    pa.field("type", pa.string()),
    pa.field("link", pa.string()),
])
 
_COLLABORATOR_STRUCT = pa.struct([
    pa.field("name", pa.string()),
    pa.field("class", pa.string()),
])
 
_REFERENCE_STRUCT = pa.struct([
    pa.field("citation", pa.string()),
    pa.field("type", pa.string()),
    pa.field("pmid", pa.int64()),
])
 
 
# ─── Per-table PyArrow schemas (mirror BigQuery types) ───────────────────────
 
TRIAL_SCHEMA = pa.schema([
    # Identification
    pa.field("nct_id", pa.string(), nullable=False),
    pa.field("brief_title", pa.string()),
    pa.field("official_title", pa.string()),
    pa.field("acronym", pa.string()),
    pa.field("organization", pa.string()),
    pa.field("organization_class", pa.string()),
    pa.field("secondary_ids", pa.list_(_SECONDARY_ID_STRUCT)),
 
    # Status / Dates
    pa.field("overall_status", pa.string()),
    pa.field("has_results", pa.bool_()),
    pa.field("start_date", pa.date32()),
    pa.field("start_date_type", pa.string()),
    pa.field("completion_date", pa.date32()),
    pa.field("primary_completion_date", pa.date32()),
    pa.field("study_first_post_date", pa.date32()),
    pa.field("last_update_post_date", pa.date32()),
 
    # Sponsor
    pa.field("lead_sponsor_name", pa.string()),
    pa.field("lead_sponsor_class", pa.string()),
    pa.field("collaborators", pa.list_(_COLLABORATOR_STRUCT)),
    pa.field("responsible_party_investigator", pa.string()),
    pa.field("responsible_party_affiliation", pa.string()),
    pa.field("responsible_party_type", pa.string()),
 
    # Design
    pa.field("study_type", pa.string()),
    pa.field("phases", pa.list_(pa.string())),
    pa.field("allocation", pa.string()),
    pa.field("intervention_model", pa.string()),
    pa.field("primary_purpose", pa.string()),
    pa.field("masking", pa.string()),
    pa.field("enrollment_count", pa.int64()),
    pa.field("enrollment_type", pa.string()),
 
    # Conditions / Keywords
    pa.field("conditions", pa.list_(pa.string())),
    pa.field("keywords", pa.list_(pa.string())),
 
    # Eligibility
    pa.field("eligibility_criteria", pa.string()),
    pa.field("min_age_raw", pa.string()),
    pa.field("max_age_raw", pa.string()),
    pa.field("min_age_years", pa.float64()),
    pa.field("max_age_years", pa.float64()),
    pa.field("sex", pa.string()),
    pa.field("std_ages", pa.list_(pa.string())),
    pa.field("healthy_volunteers", pa.bool_()),
 
    # Description
    pa.field("brief_summary", pa.string()),
    pa.field("detailed_description", pa.string()),
 
    # Oversight / Regulatory
    pa.field("is_fda_regulated_drug", pa.bool_()),
    pa.field("is_fda_regulated_device", pa.bool_()),
    pa.field("has_dmc", pa.bool_()),
 
    # Contacts / references
    pa.field("central_contacts", pa.list_(_CONTACT_STRUCT)),
    pa.field("references", pa.list_(_REFERENCE_STRUCT)),
 
    # Lineage
    pa.field("ingestion_date", pa.date32(), nullable=False),
    pa.field("ingested_at", pa.timestamp("us", tz="UTC"), nullable=False),
])
 
LOCATION_SCHEMA = pa.schema([
    pa.field("nct_id", pa.string(), nullable=False),
    pa.field("facility", pa.string()),
    pa.field("city", pa.string()),
    pa.field("state", pa.string()),
    pa.field("country", pa.string()),
    pa.field("zip", pa.string()),
    pa.field("status", pa.string()),
    pa.field("latitude", pa.float64()),
    pa.field("longitude", pa.float64()),
    pa.field("contacts", pa.list_(_CONTACT_STRUCT)),
    pa.field("ingestion_date", pa.date32(), nullable=False),
])
 
MESH_TERM_SCHEMA = pa.schema([
    pa.field("nct_id", pa.string(), nullable=False),
    pa.field("mesh_id", pa.string()),
    pa.field("mesh_term", pa.string()),
    pa.field("browse_type", pa.string()),
    pa.field("is_ancestor", pa.bool_()),
    pa.field("ingestion_date", pa.date32(), nullable=False),
])
 
ARM_GROUP_SCHEMA = pa.schema([
    pa.field("nct_id", pa.string(), nullable=False),
    pa.field("label", pa.string()),
    pa.field("type", pa.string()),
    pa.field("description", pa.string()),
    pa.field("intervention_names", pa.list_(pa.string())),
    pa.field("ingestion_date", pa.date32(), nullable=False),
])
 
INTERVENTION_SCHEMA = pa.schema([
    pa.field("nct_id", pa.string(), nullable=False),
    pa.field("type", pa.string()),
    pa.field("name", pa.string()),
    pa.field("description", pa.string()),
    pa.field("other_names", pa.list_(pa.string())),
    pa.field("arm_group_labels", pa.list_(pa.string())),
    pa.field("ingestion_date", pa.date32(), nullable=False),
])
 
OUTCOME_SCHEMA = pa.schema([
    pa.field("nct_id", pa.string(), nullable=False),
    pa.field("outcome_type", pa.string()),     # 'primary' | 'secondary' | 'other'
    pa.field("measure", pa.string()),
    pa.field("description", pa.string()),
    pa.field("time_frame", pa.string()),
    pa.field("ingestion_date", pa.date32(), nullable=False),
])
 
OFFICIAL_SCHEMA = pa.schema([
    pa.field("nct_id", pa.string(), nullable=False),
    pa.field("name", pa.string()),
    pa.field("affiliation", pa.string()),
    pa.field("role", pa.string()),
    pa.field("ingestion_date", pa.date32(), nullable=False),
])
 
TABLE_SCHEMAS: dict[str, pa.Schema] = {
    "trial": TRIAL_SCHEMA,
    "locations": LOCATION_SCHEMA,
    "mesh_terms": MESH_TERM_SCHEMA,
    "arm_groups": ARM_GROUP_SCHEMA,
    "interventions": INTERVENTION_SCHEMA,
    "outcomes": OUTCOME_SCHEMA,
    "officials": OFFICIAL_SCHEMA,
}
 
# Child tables that are 1-to-many off `trial`
_CHILD_TABLES = ("locations", "mesh_terms", "arm_groups",
                 "interventions", "outcomes", "officials")

In [42]:
# ─── Convert batched dicts -> PyArrow Tables ─────────────────────────────────
 
def studies_to_arrow_tables(
    transformed_studies: Iterable[Optional[dict]],
    ingestion_date: date,
    ingested_at: datetime,
) -> dict[str, pa.Table]:
    """
    Convert a batch of `transform_study()` outputs into one `pa.Table`
    per target table. None entries (skipped studies) are filtered out.
 
    Adds `ingestion_date` to every row, plus `ingested_at` on `trial`.
    """
    buckets: dict[str, list[dict]] = {k: [] for k in TABLE_SCHEMAS}
 
    for study in transformed_studies:
        if not study:
            continue
 
        trial_row = dict(study["trial"])
        trial_row["ingestion_date"] = ingestion_date
        trial_row["ingested_at"] = ingested_at
        buckets["trial"].append(trial_row)
 
        for key in _CHILD_TABLES:
            for row in (study.get(key) or []):
                row = dict(row)
                row["ingestion_date"] = ingestion_date
                buckets[key].append(row)
 
    tables: dict[str, pa.Table] = {}
    for name, rows in buckets.items():
        schema = TABLE_SCHEMAS[name]
        # from_pylist + explicit schema correctly handles:
        #  - nested structs / lists
        #  - missing keys -> null
        #  - datetime.date -> date32, tz-aware datetime -> timestamp[us, UTC]
        tables[name] = pa.Table.from_pylist(rows, schema=schema)
    return tables

In [43]:
# ─── Upload PyArrow Tables to GCS as Parquet ─────────────────────────────────
 
def write_tables_to_gcs(
    tables: dict[str, pa.Table],
    bucket_name: str,
    prefix: str,
    ingestion_date: date,
    batch_id: Optional[str] = None,
    compression: str = "snappy",
    storage_client: Optional[storage.Client] = None,
) -> dict[str, str]:
    """
    Write each `pa.Table` to GCS as a single Parquet object under a
    Hive-style date-partitioned path.
 
    Returns: {table_name: gs:// URI}. Empty tables are skipped (no empty
    Parquet files are written).
 
    Compression options: 'snappy' (BQ default, fastest), 'zstd' (smaller),
    'gzip' (compatibility). Stick with snappy unless storage cost dominates.
    """
    if storage_client is None:
        storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
 
    if batch_id is None:
        batch_id = uuid.uuid4().hex[:12]
 
    written: dict[str, str] = {}
    for table_name, table in tables.items():
        if table.num_rows == 0:
            logger.debug("Skipping empty table %s (no rows in batch)", table_name)
            continue
        blob_path = (
            f"{prefix.rstrip('/')}/{table_name}/"
            f"ingestion_date={ingestion_date.isoformat()}/"
            f"batch_{batch_id}.parquet"
        )
        uri = f"gs://{bucket_name}/{blob_path}"
        written[table_name] = uri
        
        pq.write_table(
            table,
            uri,
            compression=compression,         # Standard efficient compression
            use_dictionary=True,
            write_statistics=True,
            row_group_size=50_000,
        )
        logger.info("Wrote %d rows -> %s", table.num_rows, uri)
 
    return written
 
 

In [44]:
# ─── End-to-end convenience ──────────────────────────────────────────────────
 
ingested_at = datetime.now(timezone.utc)
def upload_studies_batch(
    transformed_studies: list[Optional[dict]],
    bucket_name: str,
    prefix: str = "clinical_trials/staging",
    ingestion_date: Optional[date] = None,
    storage_client: Optional[storage.Client] = None,
    compression: str = "snappy",
) -> dict[str, str]:
    """
    One-call helper: takes a batch of `transform_study()` outputs and
    writes all seven tables to GCS as Parquet.
 
    Returns: {table_name: gs:// URI}.
 
    Recommended batch size: 5,000 - 50,000 studies per call. The 'trial'
    rows are wide; the child tables (locations, mesh_terms) can be 10-50x
    that count. Larger batches = fewer, bigger Parquet files = faster
    BQ loads. Too large = OOM in the worker.
    """
    ingested_at = datetime.now(timezone.utc)
    if ingestion_date is None:
        ingestion_date = ingested_at.date()
 
    tables = studies_to_arrow_tables(
        transformed_studies, ingestion_date, ingested_at
    )
    return write_tables_to_gcs(
        tables,
        bucket_name=bucket_name,
        prefix=prefix,
        ingestion_date=ingestion_date,
        storage_client=storage_client,
        compression=compression,
    )

In [45]:
batch: list[dict] = [transformed_study]
uris = upload_studies_batch(
                batch,
               bucket_name="bitziv_health_data",
                 prefix="clinical_trials/staging",
             )

In [46]:
from __future__ import annotations
 
import logging
import uuid
from dataclasses import dataclass
from datetime import date
from typing import Literal, Optional
 
from google.cloud import bigquery
from google.cloud.bigquery import (
    Client,
    LoadJobConfig,
    SourceFormat,
    WriteDisposition,
)
 
logger = logging.getLogger(__name__)
 
 
# ─── Per-table configuration ─────────────────────────────────────────────────
 
MergeStrategy = Literal["upsert_on_nct_id", "replace_by_nct_id"]
 
 
@dataclass(frozen=True)
class TableConfig:
    gcs_subdir: str          # subdir under prefix in GCS (matches transform output keys)
    curated_table: str       # final BQ table name (within the dataset)
    strategy: MergeStrategy
 
 
TABLE_CONFIGS: tuple[TableConfig, ...] = (
    TableConfig("trial",         "clinical_trials_gov_stg",            "upsert_on_nct_id"),
    TableConfig("locations",     "clinical_trials_locations_stg",      "replace_by_nct_id"),
    TableConfig("mesh_terms",    "clinical_trials_mesh_terms_stg",     "replace_by_nct_id"),
    TableConfig("arm_groups",    "clinical_trials_arm_groups_stg",     "replace_by_nct_id"),
    TableConfig("interventions", "clinical_trials_interventions_stg",  "replace_by_nct_id"),
    TableConfig("outcomes",      "clinical_trials_outcomes_stg",       "replace_by_nct_id"),
    TableConfig("officials",     "clinical_trials_officials_stg",      "replace_by_nct_id"),
)

In [54]:
# ─── Step 1: Load Parquet from GCS into a staging table ──────────────────────
 
def load_to_staging(
    client: Client,
    bucket: str,
    prefix: str,
    gcs_subdir: str,
    ingestion_date: date,
    project: str,
    dataset: str,
    batch_id: Optional[str] = None,
) -> str:
    """
    Load all Parquet files under
        gs://{bucket}/{prefix}/{gcs_subdir}/ingestion_date=YYYY-MM-DD/
    into a temporary BQ table, fully replacing its contents.
 
    Returns the fully-qualified staging table ID.
    """
    if batch_id is None:
        batch_id = uuid.uuid4().hex[:8]
    parquet_options = bigquery.ParquetOptions()
    parquet_options.enable_list_inference = True
  
    staging_table = (
        f"{project}.{dataset}._stg_{gcs_subdir}_{ingestion_date.strftime('%Y%m%d')}_{batch_id}"
    )
    source_uri = (
        f"gs://{bucket}/{prefix.rstrip('/')}/{gcs_subdir}/"
        f"ingestion_date={ingestion_date.isoformat()}/*.parquet"
    )
 
    job_config = LoadJobConfig(
        source_format=SourceFormat.PARQUET,
        write_disposition=WriteDisposition.WRITE_TRUNCATE,
        # Parquet carries the schema; no autodetect needed.
        parquet_options=parquet_options,
    )
 
    job = client.load_table_from_uri(source_uri, staging_table, job_config=job_config)
    result = job.result()  # waits for completion, raises on failure
 
    logger.info(
        "Loaded %s rows into %s from %s",
        result.output_rows, staging_table, source_uri,
    )
    return staging_table
 

In [48]:
# ─── Step 2a: Upsert the parent 'trial' table on nct_id ──────────────────────
 
def merge_trial(
    client: Client,
    staging_table: str,
    curated_table: str,
) -> None:
    """
    MERGE staging → curated for the parent 'trial' table.
    Latest `ingested_at` wins on key collision. Inserts new nct_ids.
    """
    # Build the UPDATE SET clause from the curated table's schema (excludes nct_id).
    table = client.get_table(curated_table)
    update_cols = [f.name for f in table.schema if f.name != "nct_id"]
    update_set = ",\n    ".join(f"`{c}` = s.`{c}`" for c in update_cols)
 
    sql = f"""
    MERGE INTO `{curated_table}` t
    USING (
      SELECT *
      FROM `{staging_table}`
      -- If the same nct_id appears multiple times in one batch (shouldn't,
      -- but guard anyway), keep only the freshest.
      QUALIFY ROW_NUMBER() OVER (
        PARTITION BY nct_id ORDER BY ingested_at DESC
      ) = 1
    ) s
    ON t.nct_id = s.nct_id
    WHEN MATCHED AND s.ingested_at > t.ingested_at THEN
      UPDATE SET
        {update_set}
    WHEN NOT MATCHED THEN
      INSERT ROW
    """
 
    job = client.query(sql)
    job.result()
    stats = job.num_dml_affected_rows
    logger.info("MERGE %s -> %s affected %s rows", staging_table, curated_table, stats)

In [49]:
# ─── Step 2b: Replace child rows by nct_id ───────────────────────────────────
 
def replace_children(
    client: Client,
    staging_table: str,
    curated_table: str,
) -> None:
    """
    For child tables (many rows per nct_id), the freshest batch entirely
    replaces prior rows for any nct_id present in the batch.
 
    Runs as a single transaction:
      DELETE FROM curated WHERE nct_id IN (SELECT nct_id FROM staging);
      INSERT INTO curated SELECT * FROM staging;
    """
    sql = f"""
    BEGIN TRANSACTION;
 
    DELETE FROM `{curated_table}`
    WHERE nct_id IN (SELECT DISTINCT nct_id FROM `{staging_table}`);
 
    INSERT INTO `{curated_table}`
    SELECT * FROM `{staging_table}`;
 
    COMMIT TRANSACTION;
    """
 
    job = client.query(sql)
    job.result()
    logger.info("Replaced child rows in %s from %s", curated_table, staging_table)

In [50]:
# ─── Cleanup ─────────────────────────────────────────────────────────────────
 
def drop_staging(client: Client, staging_table: str) -> None:
    client.delete_table(staging_table, not_found_ok=True)
    logger.debug("Dropped staging table %s", staging_table)

In [51]:
# ─── Orchestrator ────────────────────────────────────────────────────────────
 
def load_batch_to_bigquery(
    bucket: str,
    prefix: str,
    ingestion_date: date,
    project: str,
    dataset: str,
    client: Optional[Client] = None,
    drop_staging_after: bool = True,
) -> None:
    """
    Run the full Parquet → staging → curated pipeline for all seven tables
    for a given ingestion_date.
 
    Tables are processed sequentially; each landing/merge waits for the
    previous to finish. Failures raise; partial state is allowed (a failed
    child merge does not roll back a successful parent merge — each is
    its own transaction). Re-running the orchestrator for the same date
    is safe and idempotent.
    """
    if client is None:
        client = bigquery.Client(project=project)
 
    batch_id = uuid.uuid4().hex[:8]
    logger.info(
        "Starting BQ load: project=%s dataset=%s ingestion_date=%s batch_id=%s",
        project, dataset, ingestion_date, batch_id,
    )
 
    for cfg in TABLE_CONFIGS:
        curated = f"{project}.{dataset}.{cfg.curated_table}"
 
        try:
            staging = load_to_staging(
                client=client,
                bucket=bucket,
                prefix=prefix,
                gcs_subdir=cfg.gcs_subdir,
                ingestion_date=ingestion_date,
                project=project,
                dataset=dataset,
                batch_id=batch_id,
            )
        except Exception as e:
            # If there are no files for this table for this date (e.g., the
            # batch happened to contain no trials with locations), skip it.
            if "Not found" in str(e) or "does not match any" in str(e):
                logger.warning("No Parquet for %s on %s — skipping", cfg.gcs_subdir, ingestion_date)
                continue
            raise
 
        try:
            if cfg.strategy == "upsert_on_nct_id":
                merge_trial(client, staging, curated)
            else:
                replace_children(client, staging, curated)
        finally:
            if drop_staging_after:
                drop_staging(client, staging)
 
    logger.info("Completed BQ load for ingestion_date=%s", ingestion_date)

In [55]:
load_batch_to_bigquery(
    bucket="bitziv_health_data",
    prefix="clinical_trials/staging",
    ingestion_date = ingested_at.date(),
    project="data-eng-470620",
    dataset="healthcare_ai",
)

In [ ]:
from __future__ import annotations
 
import hashlib
import logging
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Iterable, Optional
 
import torch
from sentence_transformers import SentenceTransformer
 
logger = logging.getLogger(__name__)
 
 
# ─── Constants ───────────────────────────────────────────────────────────────
 
DEFAULT_MODEL = "nomic-ai/nomic-embed-text-v1.5"
EMBEDDING_DIM = 768
 
# Nomic task prefixes — must match between index and query time.
INDEX_PREFIX = "search_document: "
QUERY_PREFIX = "search_query: "
# ─── Per-field text builders ─────────────────────────────────────────────────


/Users/baljotsingh/kernvind/healtcare/ctiq/pipelines/ctiq_etl/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
 
def _title_text(trial: dict) -> Optional[str]:
    """Brief title preferred; falls back to official title."""
    return trial.get("brief_title") or trial.get("official_title")

def _summary_text(trial: dict) -> Optional[str]:
    """
    Composite 'summary' document for general discovery and 'find similar trials'.
 
    Deliberately excludes enums (status, phase, allocation, masking,
    primary_purpose, study_type). Those are SQL pre-filters at query time;
    encoding them into the embedding wastes capacity and forces re-embed
    when status changes from RECRUITING -> COMPLETED.
    """
    parts: list[str] = []
    if trial.get("official_title"):
        parts.append(f"TITLE: {trial['official_title']}")
    if trial.get("conditions"):
        parts.append(f"CONDITIONS: {', '.join(trial['conditions'])}")
    if trial.get("keywords"):
        parts.append(f"KEYWORDS: {', '.join(trial['keywords'])}")
    if trial.get("brief_summary"):
        parts.append(f"SUMMARY:\n{trial['brief_summary']}")
    if trial.get("detailed_description"):
        parts.append(f"DESCRIPTION:\n{trial['detailed_description']}")
    return "\n\n".join(parts) if parts else None
 
 
def _eligibility_text(trial: dict) -> Optional[str]:
    """
    Raw eligibility criteria as a single chunk.
 
    v0 limitation: long eligibility text averaged into one vector hurts
    matching quality. When you add the LLM splitter, replace this with a
    function that returns list[str] of individual inclusion/exclusion items;
    no other code in this module needs to change.
    """
    text = trial.get("eligibility_criteria")
    return text or None
 
 
def _outcome_text(outcome: dict) -> str:
    parts = []
    if outcome.get("measure"):
        parts.append(outcome["measure"])
    if outcome.get("description"):
        parts.append(outcome["description"])
    if outcome.get("time_frame"):
        parts.append(f"(Time frame: {outcome['time_frame']})")
    return " ".join(parts).strip()
 
 
def _intervention_text(intervention: dict) -> str:
    parts = []
    if intervention.get("name"):
        parts.append(intervention["name"])
    if intervention.get("other_names"):
        parts.append(f"(also known as: {', '.join(intervention['other_names'])})")
    if intervention.get("description"):
        parts.append(intervention["description"])
    return " ".join(parts).strip()
 
 
def _hash_text(text: str) -> str:
    """SHA-256 of raw chunk text (no prefix) for change detection."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

In [58]:
# ─── Chunk extraction ────────────────────────────────────────────────────────
 
@dataclass
class EmbeddingChunk:
    """One text chunk awaiting embedding."""
    nct_id: str
    field_type: str
    chunk_id: int
    chunk_text: str  # without the task prefix
 
 
def extract_chunks(transformed: dict) -> list[EmbeddingChunk]:
    """
    Produce all chunks to embed for one trial.
 
    Input is the dict returned by transform_study(), with keys
    {'trial', 'outcomes', 'interventions', ...}.
    """
    trial = transformed["trial"]
    nct_id = trial["nct_id"]
    chunks: list[EmbeddingChunk] = []
 
    if (title := _title_text(trial)):
        chunks.append(EmbeddingChunk(nct_id, "title", 0, title))
 
    if (summary := _summary_text(trial)):
        chunks.append(EmbeddingChunk(nct_id, "summary", 0, summary))
 
    if (elig := _eligibility_text(trial)):
        # When you add LLM splitting, replace this single-chunk block with a
        # loop over the split items, each as its own chunk_id.
        chunks.append(EmbeddingChunk(nct_id, "eligibility", 0, elig))
 
    for i, outcome in enumerate(transformed.get("outcomes") or []):
        text = _outcome_text(outcome)
        if text:
            chunks.append(EmbeddingChunk(nct_id, "outcome", i, text))
 
    for i, intervention in enumerate(transformed.get("interventions") or []):
        text = _intervention_text(intervention)
        if text:
            chunks.append(EmbeddingChunk(nct_id, "intervention", i, text))
 
    return chunks

In [68]:
# ─── Embedder ────────────────────────────────────────────────────────────────
 
class TrialEmbedder:
    """
    Sentence-Transformers wrapper specialized for clinical trial chunks.
 
    The model is loaded once at __init__ and reused across calls. GPU is
    auto-selected if available; pass device='cpu' to force CPU.
    """
 
    def __init__(
        self,
        model_name: str = DEFAULT_MODEL,
        device: Optional[str] = None,
        max_seq_length: int = 8192,
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info("Loading embedding model %s on %s", model_name, device)
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device,
        )
        self.model.max_seq_length = max_seq_length
        self.model_name = model_name
        # Use the HF model id as the version string. If you fine-tune or
        # change the truncation dim, append a suffix here so old/new vectors
        # don't get mixed in BQ.
        self.model_version = model_name
 
    def embed_chunks(
        self,
        chunks: list[EmbeddingChunk],
        batch_size: int = 32,
        show_progress: bool = False,
    ) -> list[dict]:
        """
        Embed a batch of chunks and return rows for the embeddings table.
 
        Applies the index-time prefix automatically. Output vectors are
        L2-normalized so cosine similarity equals dot product downstream.
        """
        if not chunks:
            return []
 
        prefixed_texts = [INDEX_PREFIX + c.chunk_text for c in chunks]
 
        vectors = self.model.encode(
            prefixed_texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            show_progress_bar=show_progress,
            normalize_embeddings=True,
        )
 
        now = datetime.now(timezone.utc)
        rows: list[dict] = []
        for chunk, vec in zip(chunks, vectors):
            rows.append({
                "nct_id": chunk.nct_id,
                "field_type": chunk.field_type,
                "chunk_id": chunk.chunk_id,
                "chunk_text": chunk.chunk_text,
                "source_hash": _hash_text(chunk.chunk_text),
                "embedding": vec.tolist(),
                "model_version": self.model_version,
                "embedded_at": now,
           })
        return rows
 
    def embed_trials(
        self,
        transformed_studies: Iterable[dict],
        batch_size: int = 32,
        show_progress: bool = False,
    ) -> list[dict]:
        """Convenience: extract chunks from each study and embed them all."""
        all_chunks: list[EmbeddingChunk] = []
        for transformed in transformed_studies:
            if not transformed:
                continue
            all_chunks.extend(extract_chunks(transformed))
        return self.embed_chunks(
            all_chunks,
            batch_size=batch_size,
            show_progress=show_progress,
        )
 
    def embed_query(self, query: str) -> list[float]:
        """
        Embed a user query for retrieval. Uses the 'search_query: ' prefix —
        critical for alignment with index-side embeddings.
        """
        vec = self.model.encode(
            QUERY_PREFIX + query,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vec.tolist()

In [ ]:
embedder = TrialEmbedder()
chunks = extract_chunks(transformed_study)

prefixed_texts = [INDEX_PREFIX + c.chunk_text for c in chunks]
print(prefixed_texts)
all_chunks = [chunks]
rows = embedder.embed_chunks(chunks, batch_size=64, show_progress=True)
rows

In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "mps" if torch.backends.mps.is_built() else "cpu"
device

'mps'